# Step 6: Machine Learning Model
• Choose a suitable machine learning approach:
• Split the data into training and testing sets.
• Train the model and evaluate its performance using appropriate metrics (e.g., accuracy, RMSE, etc.).
Deliverable: Trained machine learning model and evaluation results.

### Machine Learning Model: Predicting Fire Risk Levels

This cell implements the machine learning workflow for predicting the fire risk level of each incident's district. It prepares the feature set and target variable, splits the data into training and testing sets, trains a Random Forest classifier, and evaluates the model's performance using accuracy, confusion matrix, and classification report. The results provide insight into how well the engineered features can predict fire risk across San Francisco.

In [ ]:
import pandas as pd

fire_incidents_df = pd.read_csv('feature_engineered_fire_incidents.csv')
fire_incidents_df.head()

In [ ]:
# Data loading
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Preprocessing: Prepare features and target (add neighborhood_district one-hot columns)
feature_columns = ['dist_to_fire_station_m'] + [col for col in fire_incidents_df.columns if col.startswith('neigh_')]
X_features = fire_incidents_df[feature_columns].copy()
y_risk_level = fire_incidents_df['risk_level']

print('Features used for modeling:')
print(X_features.head())
print('\nTarget variable:')
print(y_risk_level.value_counts())

In [ ]:
# Train-test split
X_train_features, X_test_features, y_train_risk_level, y_test_risk_level = train_test_split(
    X_features, y_risk_level, test_size=0.3, random_state=42, stratify=y_risk_level
)
print(f'\nTraining samples: {len(X_train_features)}, Test samples: {len(X_test_features)}')

In [ ]:
# Model training
fire_risk_rf_classifier = RandomForestClassifier(n_estimators=50, random_state=42)
fire_risk_rf_classifier.fit(X_train_features, y_train_risk_level)
print('\nRandom Forest model trained.')

In [ ]:
# Evaluation metrics
y_pred_risk_level = fire_risk_rf_classifier.predict(X_test_features)
test_accuracy = accuracy_score(y_test_risk_level, y_pred_risk_level)
print(f'\nTest Accuracy: {test_accuracy:.3f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test_risk_level, y_pred_risk_level))
print('\nClassification Report:')
print(classification_report(y_test_risk_level, y_pred_risk_level, zero_division=0))

In [7]:
# Export incident-level predictions for visualization in EDA notebook
test_incidents_df = fire_incidents_df.loc[X_test_features.index].copy()
test_incidents_df['predicted_risk_level'] = y_pred_risk_level
test_incidents_df.to_csv('incident_predictions_with_risk.csv', index=False)